# DVAE Baseline

Ce notebook reproduit la baseline DVAE sur 20NG, AGNews, IMDB et YahooAnswer.
Le protocole suit trois etapes : preparation de l'environnement, entrainement avec export des matrices, puis evaluation des metriques TD, Purity, NMI et C_V.


In [ ]:
import sys
import subprocess
import importlib.util


def ensure_package(import_name: str, pip_spec: str):
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_spec}")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            pip_spec,
        ])


# Kaggle-safe: do not force-upgrade torch/cuda.
ensure_package("numpy", "numpy")
ensure_package("scipy", "scipy")
ensure_package("pandas", "pandas")
ensure_package("sklearn", "scikit-learn")
ensure_package("yaml", "pyyaml")
ensure_package("gensim", "gensim")
ensure_package("tqdm", "tqdm")
ensure_package("torch", "torch")

print("Dependencies check done")


In [ ]:
import numpy as np
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
import scipy.io
import os
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import normalized_mutual_info_score
import matplotlib.pyplot as plt


def detect_ppd_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, cwd / 'Projet_PPD' / 'Projet_PPD', cwd / 'Projet_PPD']
    ordered = []
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            continue
        key = str(resolved)
        if key not in seen:
            seen.add(key)
            ordered.append(resolved)
    for candidate in ordered:
        if (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Impossible de localiser un dossier contenant data/ et output/.')


PPD_ROOT = detect_ppd_root()

SEED = 42
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

hiddensize = 256
batchsize = 200
TOPN = 15
DATASETS = ['20NG', 'AGNews', 'IMDB', 'YahooAnswer']
KVALUES = [50, 100]

DATAROOT = str(PPD_ROOT / 'data')
DVAEROOT = str(PPD_ROOT / 'output' / 'DVAE')
os.makedirs(DVAEROOT, exist_ok=True)
print(f'DATAROOT: {DATAROOT}')
print(f'DVAEROOT: {DVAEROOT}')


In [ ]:
import scipy.sparse as sp


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def locate_first_existing(base_path, filenames):
    for filename in filenames:
        candidate = os.path.join(base_path, filename)
        if os.path.exists(candidate):
            return candidate
    return None


def load_dataset(name):
    path = os.path.join(DATAROOT, name)
    print(f'Chargement du dataset {name}: {path}')

    train_files = ['trainbow.npz', 'train.npz', 'train_bow.npz', 'trainbow.npy']
    test_files = ['testbow.npz', 'test.npz', 'test_bow.npz', 'testbow.npy']
    vocab_files = ['vocab.txt', 'vocabulary.txt']

    train_file = locate_first_existing(path, train_files)
    if train_file is None:
        raise FileNotFoundError(f'Aucun fichier train compatible trouve pour {name}.')

    test_file = locate_first_existing(path, test_files)
    if test_file is None:
        raise FileNotFoundError(f'Aucun fichier test compatible trouve pour {name}.')

    vocab_file = locate_first_existing(path, vocab_files)

    if train_file.endswith('.npz'):
        train = sp.load_npz(train_file).toarray()
    else:
        train = np.load(train_file)

    if test_file.endswith('.npz'):
        test = sp.load_npz(test_file).toarray()
    else:
        test = np.load(test_file)

    if vocab_file is not None:
        with open(vocab_file, encoding='utf-8') as f:
            vocab = [w.strip() for w in f if w.strip()]
    else:
        vocab = [f'word_{i}' for i in range(train.shape[1])]

    print(f'Dataset {name}: train={train.shape}, test={test.shape}, vocab={len(vocab)}')
    return train, test, vocab


def topic_diversity(beta, vocab, topn=15):
    all_words = []
    for k in range(beta.shape[0]):
        topidx = np.argsort(beta[k, :])[-topn:]
        all_words.extend([vocab[i] for i in topidx])
    return len(set(all_words)) / len(all_words)


def purity_score(y_true, y_pred):
    y_true_int = y_true.astype(np.int64)
    y_pred_int = y_pred.astype(np.int64)

    purity = 0
    for c in np.unique(y_pred_int):
        idx = np.where(y_pred_int == c)[0]
        if len(idx) == 0:
            continue
        counts = np.bincount(y_true_int[idx])
        purity += counts.max()
    return purity / len(y_true)


In [ ]:
class DVAEBaseline(nn.Module):
    def __init__(self, vocabsize, numtopics, hiddensize, prioralpha, dropout=0.2, klweight=1.0):
        super().__init__()
        self.klweight = klweight
        self.numtopics = numtopics
        
        self.encoder = nn.Sequential(
            nn.Linear(vocabsize, hiddensize),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hiddensize, numtopics)
        )
        
        self.beta = nn.Parameter(torch.Tensor(numtopics, vocabsize))
        nn.init.xavier_uniform_(self.beta)
        self.decoderbn = nn.BatchNorm1d(vocabsize, affine=False)
        self.register_buffer('prioralpha', torch.full((numtopics,), prioralpha))
    
    def encode(self, x):
        raw = self.encoder(x)
        alpha = F.softplus(raw + 1e-6)
        return torch.clamp(alpha, min=1e-3, max=50.0)
    
    def sample_theta(self, alpha):
        if self.training:
            concentration = torch.clamp(alpha, min=1e-2)
            rate = torch.ones_like(concentration)
            gamma = torch.distributions.Gamma(concentration, rate)
            z = gamma.rsample()
            theta = z / (z.sum(dim=1, keepdim=True) + 1e-12)
        else:
            theta = alpha / (alpha.sum(dim=1, keepdim=True) + 1e-12)
        return torch.clamp(theta, min=1e-12)
    
    def decode(self, theta):
        res = torch.matmul(theta, self.beta)
        res = self.decoderbn(res)
        return F.log_softmax(res, dim=1)
    
    def forward(self, x_raw):
        # Normalisation
        x_sum = x_raw.sum(dim=1, keepdim=True)
        x_norm = x_raw / torch.clamp(x_sum, min=1e-12)
        
        # Forward pass
        alpha = self.encode(x_norm)
        theta = self.sample_theta(alpha)
        logprobs = self.decode(theta)
        recon = -(x_raw * logprobs).sum(dim=1).mean()
        
        # KL SIMPLIFIÉE (PAS D'ERREUR DE DIMENSION !)
        # KL SIMPLIFIÉE (sans warning)
        mean_alpha = alpha.mean(dim=0)  # (K,)
        prior_logit = self.prioralpha.detach().clone().to(alpha.device)  # ✅ FIX ICI
        kl = F.kl_div(
            F.log_softmax(prior_logit.unsqueeze(0), dim=0),
            F.softmax(mean_alpha, dim=0).unsqueeze(0),
            reduction='batchmean'
        )

        
        return recon + self.klweight * kl
    
    def get_beta(self):
        self.eval()
        with torch.no_grad():
            ident = torch.eye(self.numtopics).to(self.beta.device)
            logits = self.decoderbn(torch.matmul(ident, self.beta))
            return F.softmax(logits, dim=1)
    
    def get_theta(self, x_raw):
        self.eval()
        with torch.no_grad():
            x_sum = x_raw.sum(dim=1, keepdim=True)
            x_norm = x_raw / torch.clamp(x_sum, min=1e-12)
            alpha = self.encode(x_norm)
            return alpha / (alpha.sum(dim=1, keepdim=True) + 1e-12)


In [ ]:
def train_dvae_pipeline():
    results = []
    timing_rows = []

    for dataset in DATASETS:
        print(f"\n{'=' * 50}")
        print(f'Traitement de {dataset}')
        print(f"{'=' * 50}")

        if dataset == '20NG':
            hp_50 = {'dropout': 0.98, 'klweight': 50.0, 'prioralpha': 200.0, 'epochs': 8}
            hp_100 = {'dropout': 0.96, 'klweight': 45.0, 'prioralpha': 150.0, 'epochs': 12}
        elif dataset == 'AGNews':
            hp_50 = {'dropout': 0.92, 'klweight': 35.0, 'prioralpha': 120.0, 'epochs': 10}
            hp_100 = {'dropout': 0.88, 'klweight': 30.0, 'prioralpha': 100.0, 'epochs': 15}
        elif dataset == 'IMDB':
            hp_50 = {'dropout': 0.99, 'klweight': 60.0, 'prioralpha': 250.0, 'epochs': 6}
            hp_100 = {'dropout': 0.97, 'klweight': 55.0, 'prioralpha': 200.0, 'epochs': 10}
        else:
            hp_50 = {'dropout': 0.95, 'klweight': 42.0, 'prioralpha': 160.0, 'epochs': 9}
            hp_100 = {'dropout': 0.92, 'klweight': 38.0, 'prioralpha': 130.0, 'epochs': 14}

        trainbow, testbow, vocab = load_dataset(dataset)
        V = trainbow.shape[1]
        ytrue_path = os.path.join(DATAROOT, dataset, 'test_labels.txt')

        if not os.path.exists(ytrue_path):
            print(f'test_labels.txt absent pour {dataset}; une cible vide sera utilisee.')
            ytrue = np.zeros(len(testbow))
        else:
            ytrue = np.loadtxt(ytrue_path, dtype=int)

        for K in KVALUES:
            print(f'\nK={K}')
            hp = hp_50 if K == 50 else hp_100
            print(f'Hyperparametres: {hp}')

            set_seed(SEED)
            total_start = time.perf_counter()
            model = DVAEBaseline(V, K, hiddensize, hp['prioralpha'], hp['dropout'], hp['klweight']).to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

            Xtrain = torch.tensor(trainbow, dtype=torch.float32).to(device)
            loader = DataLoader(TensorDataset(Xtrain), batch_size=batchsize, shuffle=True)

            train_start = time.perf_counter()
            model.train()
            for epoch in range(hp['epochs']):
                for batch in loader:
                    batch = batch[0].to(device)
                    optimizer.zero_grad()
                    loss = model(batch)
                    loss.backward()
                    optimizer.step()
            train_seconds = time.perf_counter() - train_start

            model.eval()
            with torch.no_grad():
                beta = model.get_beta().cpu().numpy()
                theta = model.get_theta(torch.tensor(testbow, dtype=torch.float32).to(device)).cpu().numpy()
                ypred = theta.argmax(axis=1)
                td = topic_diversity(beta, vocab, TOPN)
                purity = purity_score(ytrue, ypred)
                nmi = normalized_mutual_info_score(ytrue, ypred)

            total_seconds = time.perf_counter() - total_start

            results.append({'Dataset': dataset, 'K': K, 'TD': td, 'Purity': purity, 'NMI': nmi})
            print(f'K={K}: TD={td:.4f}, Purity={purity:.4f}, NMI={nmi:.4f}')

            timing_row = {
                'model': 'DVAE', 'dataset': dataset, 'K': int(K), 'seed': int(SEED), 'device': str(device),
                'phase': 'train', 'train_seconds': float(train_seconds), 'total_seconds': float(total_seconds),
                'train_minutes': float(train_seconds / 60.0), 'total_minutes': float(total_seconds / 60.0),
            }
            timing_rows.append(timing_row)

            ds_out = os.path.join(DVAEROOT, dataset)
            os.makedirs(ds_out, exist_ok=True)
            mat_path = os.path.join(ds_out, f'{dataset}_DVAE_K{K}_seed{SEED}.mat')
            scipy.io.savemat(mat_path, {'beta': beta, 'test_theta': theta})
            pd.DataFrame([timing_row]).to_csv(os.path.join(ds_out, f'{dataset}_DVAE_K{K}_seed{SEED}_timing.csv'), index=False)
            print(f'Saved: {mat_path}')

    df = pd.DataFrame(results).sort_values(['Dataset', 'K'])
    display(df.round(4))
    results_csv = os.path.join(DVAEROOT, 'DVAE_RESULTS.csv')
    df.to_csv(results_csv, index=False)
    print(f'Resultats sauvegardes: {results_csv}')

    if timing_rows:
        df_timing = pd.DataFrame(timing_rows).sort_values(['dataset', 'K']).reset_index(drop=True)
        display(df_timing[['dataset', 'K', 'device', 'train_seconds', 'total_seconds']])
        time_csv = os.path.join(DVAEROOT, 'DVAE_training_times.csv')
        df_timing.to_csv(time_csv, index=False)
        print(f'Saved training times: {time_csv}')

    return df


results_df = train_dvae_pipeline()


In [ ]:
print('Calcul de la coherence C_V avec Gensim pour DVAE')
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
import numpy as np
import scipy.io
import os
from tqdm.auto import tqdm

DVAEROOT = os.path.join(os.path.dirname(DATAROOT), 'output', 'DVAE')
cv_dvae_results = []
TOPN = 15
K_VALUES = [50, 100]

for dataset in tqdm(DATASETS, desc='DVAE C_V Gensim'):
    train_bow, _, vocab = load_dataset(dataset)
    texts = []
    for doc in train_bow[:10000]:
        doc_words = [vocab[i] for i, freq in enumerate(doc) if freq > 0]
        if len(doc_words) >= 2:
            texts.append(doc_words[:100])

    dictionary = Dictionary(texts)
    corpus = [dictionary.doc2bow(text) for text in texts]
    print(f'{dataset}: {len(texts)} documents tokenises, {len(dictionary)} tokens')

    for K in K_VALUES:
        mat_paths = [
            f'{DVAEROOT}/{dataset}/{dataset}_DVAE_K{K}_seed42.mat',
            f'{DVAEROOT}/{dataset}_K{K}_DVAE.mat',
            f'{DVAEROOT}/dvae_{dataset}_K{K}.mat',
            f'{DVAEROOT}/{dataset}_K{K}.mat',
        ]

        mat_path = next((p for p in mat_paths if os.path.exists(p)), None)
        if mat_path is None:
            print(f'Fichier beta introuvable pour {dataset} K={K}')
            continue

        data = scipy.io.loadmat(mat_path)
        beta = None
        for key in ['beta', 'betas', 'topic_word', 'beta_dvae']:
            if key in data and data[key].size > 0:
                beta = data[key]
                break

        if beta is None:
            print(f'Beta introuvable dans {mat_path}')
            continue

        topics = []
        for topic_id in range(min(K, beta.shape[0])):
            idx = np.argsort(beta[topic_id])[-TOPN:][::-1]
            topic_words = [vocab[i] for i in idx if i < len(vocab) and vocab[i] in dictionary.token2id]
            if len(topic_words) >= 2:
                topics.append(topic_words[:TOPN])

        if len(topics) < max(1, int(0.8 * K)):
            print(f'Nombre de topics valides insuffisant pour {dataset} K={K}')
            continue

        cm = CoherenceModel(
            topics=topics,
            texts=texts,
            corpus=corpus,
            dictionary=dictionary,
            coherence='c_v',
        )

        cv_score = float(cm.get_coherence())
        cv_dvae_results.append({'Dataset': dataset, 'K': K, 'C_V': round(cv_score, 4)})
        print(f'{dataset:12} K={K:3d} | C_V={cv_score:.4f}')

if cv_dvae_results:
    df = pd.DataFrame(cv_dvae_results)
    print('\nTableau DVAE C_V Gensim')
    print(df.pivot(index='Dataset', columns='K', values='C_V').round(4))
    df.to_csv(f'{DVAEROOT}/DVAE_CV_OFFICIEL.csv', index=False)
    print('DVAE_CV_OFFICIEL.csv sauvegarde')
else:
    print('Aucun score C_V n a pu etre calcule pour DVAE.')


In [ ]:
import pandas as pd

# 1) C_V DVAE (tableau 1)
dvae_cv = {
    "20NG":        {50: 0.5414, 100: 0.5334},
    "AGNews":      {50: 0.4128, 100: 0.4206},
    "IMDB":        {50: 0.3880, 100: 0.3853},
    "YahooAnswer": {50: 0.6247, 100: 0.5830},
}

# 2) TD / Purity / NMI DVAE (dataframe que tu as déjà)
#   Dataset   K      TD   Purity   NMI
#   20NG     50   0.7320  0.0823  0.0356
dvae_other = pd.DataFrame({
    "Dataset": ["20NG","20NG","AGNews","AGNews","IMDB","IMDB","YahooAnswer","YahooAnswer"],
    "K":       [50,100, 50,100, 50,100, 50,100],
    "TD":      [0.7320,0.5733, 0.7720,0.6513, 0.5613,0.4040, 0.7467,0.6067],
    "Purity":  [0.0823,0.0535, 0.2736,0.2952, 0.5000,0.5445, 0.1360,0.1072],
    "NMI":     [0.0356,0.0013, 0.0228,0.0265, 0.0000,0.0188, 0.0164,0.0025],
})

# 3) On construit un seul tableau DVAE complet
rows = []
for ds in ["20NG","AGNews","IMDB","YahooAnswer"]:
    for K in [50,100]:
        base = dvae_other[(dvae_other["Dataset"] == ds) & (dvae_other["K"] == K)].iloc[0]
        row = {
            "Dataset": ds,
            "K": K,
            "C_V": dvae_cv[ds][K],
            "TD": base["TD"],
            "Purity": base["Purity"],
            "NMI": base["NMI"],
        }
        rows.append(row)

df_dvae = pd.DataFrame(rows).set_index(["Dataset","K"])

display(
    df_dvae.style
        .format("{:.4f}")
        .set_caption("DVAE — C_V GENSIM + TD + Purity + NMI")
)


In [ ]:
import pandas as pd

# Données Papier extraites des captures
paper_dvae = {
    "20NG":   {"K50": {"CV": 0.331, "TD": 0.598, "Purity": 0.087, "NMI": 0.018},
               "K100": {"CV": 0.372, "TD": 0.658, "Purity": 0.104, "NMI": 0.035}},
    "IMDB":   {"K50": {"CV": 0.294, "TD": 0.050, "Purity": 0.517, "NMI": 0.000},
               "K100": {"CV": 0.290, "TD": 0.229, "Purity": 0.525, "NMI": 0.001}},
    "Yahoo":  {"K50": {"CV": 0.338, "TD": 0.674, "Purity": 0.171, "NMI": 0.030},
               "K100": {"CV": 0.376, "TD": 0.589, "Purity": 0.202, "NMI": 0.057}},
    "AGNews": {"K50": {"CV": 0.419, "TD": 0.347, "Purity": 0.713, "NMI": 0.219},
               "K100": {"CV": 0.298, "TD": 0.113, "Purity": 0.407, "NMI": 0.030}},
}

# Données de Reproduction arrondies à 3 chiffres
repro_dvae = {
    "20NG":   {"K50": {"CV": 0.541, "TD": 0.732, "Purity": 0.082, "NMI": 0.036},
               "K100": {"CV": 0.533, "TD": 0.573, "Purity": 0.054, "NMI": 0.001}},
    "AGNews": {"K50": {"CV": 0.413, "TD": 0.772, "Purity": 0.274, "NMI": 0.023},
               "K100": {"CV": 0.421, "TD": 0.651, "Purity": 0.295, "NMI": 0.027}},
    "IMDB":   {"K50": {"CV": 0.388, "TD": 0.561, "Purity": 0.500, "NMI": 0.000},
               "K100": {"CV": 0.385, "TD": 0.404, "Purity": 0.545, "NMI": 0.019}},
    "Yahoo":  {"K50": {"CV": 0.625, "TD": 0.747, "Purity": 0.136, "NMI": 0.016},
               "K100": {"CV": 0.583, "TD": 0.607, "Purity": 0.107, "NMI": 0.003}},
}

data = []
metrics = ["CV", "TD", "Purity", "NMI"]

for ds in ["20NG", "IMDB", "Yahoo", "AGNews"]:
    for k_key in ["K50", "K100"]:
        row = {"Dataset": ds, "K": int(k_key[1:])}
        for m in metrics:
            p_val = paper_dvae[ds][k_key][m]
            r_val = repro_dvae[ds][k_key][m]
            row[f"{m}_Papier"] = p_val
            row[f"{m}_Repro"] = r_val
            row[f"{m}_Ecart"] = round(r_val - p_val, 3)
        data.append(row)

df = pd.DataFrame(data).set_index(["Dataset", "K"])

def color_ecart(val):
    if abs(val) < 0.02: return "background-color: #c6efce; color: #276221"
    if abs(val) < 0.05: return "background-color: #ffeb9c; color: #9c6500"
    return "background-color: #ffc7ce; color: #9c0006"

ecart_cols = [c for c in df.columns if "Ecart" in c]

display(df.style
    .format("{:.3f}")
    .map(color_ecart, subset=ecart_cols)
    .set_caption("DVAE — Papier vs Repro (Arrondi 3 décimales)")
)